# IMDb Movie Recommendation System

## 3. Storyline-Based Recommendation System

This notebook builds the core recommendation engine for the IMDb Movie Recommendation System.

The system uses a **content-based recommendation approach**. Movie storylines are converted into numerical representations using **TF-IDF (Term Frequency-Inverse Document Frequency)**, and **Cosine Similarity** is used to measure the similarity between a user's input storyline and the available movie storylines.

### Recommendation Workflow

1. Load the preprocessed movie dataset.
2. Validate the processed storyline data.
3. Create TF-IDF representations of movie storylines.
4. Transform a user's storyline into the same TF-IDF feature space.
5. Calculate cosine similarity between the user query and all movies.
6. Rank movies by similarity score.
7. Return the top 5 recommended movies.

### Recommendation Type

This implementation uses **content-based filtering**, meaning recommendations are generated from the textual characteristics of movie storylines rather than user ratings, viewing history, or collaborative filtering.

In [49]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np


# Locate project root dynamically
current_dir = Path.cwd()

project_root = next(
    (
        path
        for path in [current_dir, *current_dir.parents]
        if (path / "src" / "config.py").exists()
    ),
    None
)

if project_root is None:
    raise FileNotFoundError(
        "Project root could not be located."
    )

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [50]:
from src.config import PROCESSED_DATA_PATH, MODELS_DIR

print("Project root:", project_root)
print("Processed dataset:", PROCESSED_DATA_PATH)
print("Models directory:", MODELS_DIR)

Project root: g:\Projects\IMDb_Movie_Recommendation_System
Processed dataset: G:\Projects\IMDb_Movie_Recommendation_System\data\IMDBRecSys_Processed.csv
Models directory: G:\Projects\IMDb_Movie_Recommendation_System\models


In [51]:
df = pd.read_csv(PROCESSED_DATA_PATH)

df.head()

,Movie_Title,Storyline_Raw,Storyline_Processed
0,The Fall Guy,"A stuntman, fresh off an almost career-ending ...",stuntman fresh almost career ending accident t...
1,The Substance,A fading celebrity takes a black-market drug: ...,fading celebrity takes black market drug cell ...
2,The Life of Chuck,"A life-affirming, genre-bending story about th...",life affirming genre bending story three chapt...
3,Abigail,After a group of criminals kidnap the ballerin...,group criminals kidnap ballerina daughter powe...
4,The Ministry of Ungentlemanly Warfare,The British military recruits a small group of...,british military recruits small group highly s...


In [52]:
print(f"Rows    : {len(df):,}")
print(f"Columns : {len(df.columns)}")

Rows    : 6,021
Columns : 3


## 1. Input Validation

The recommendation engine will use `Storyline_Processed`, which contains the normalized storyline text produced during the NLP preprocessing stage.

Before vectorization, the column is checked for missing or empty values.

In [53]:
df["Storyline_Processed"].isna().sum()

np.int64(0)

In [54]:
(
    df["Storyline_Processed"]
    .str.strip()
    .eq("")
    .sum()
)

np.int64(0)

## 2. TF-IDF Vectorization

TF-IDF is used to represent each movie storyline as a numerical vector.

The technique gives greater importance to words that are relatively important within a storyline while reducing the influence of words that occur frequently across many storylines.

The resulting vectors will be used as the feature representation for similarity calculation.

In [55]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [56]:
tfidf = TfidfVectorizer(
    stop_words="english"
)

In [57]:
tfidf_matrix = tfidf.fit_transform(
    df["Storyline_Processed"]
)

In [58]:
print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (6021, 18700)


In [59]:
print("Matrix type:", type(tfidf_matrix))
print("Matrix format:", tfidf_matrix.format)

Matrix type: <class 'scipy.sparse._csr.csr_matrix'>
Matrix format: csr


In [60]:
feature_names = tfidf.get_feature_names_out()

print("Vocabulary size:", len(feature_names))

Vocabulary size: 18700


In [61]:
feature_names[:50]

array(['aadhi', 'aadi', 'aagje', 'aalparambil', 'aamina', 'aan', 'aarjav',
       'aarohi', 'aaron', 'abah', 'abandon', 'abandoned', 'abandoning',
       'abandonment', 'abandons', 'abbas', 'abbess', 'abbeville', 'abbey',
       'abbie', 'abbot', 'abbruzzi', 'abby', 'abd', 'abduct', 'abducted',
       'abductee', 'abduction', 'abductor', 'abducts', 'abdul', 'abe',
       'abel', 'abertzale', 'abhimanyu', 'abhiram', 'abhirup', 'abi',
       'abiding', 'abigail', 'abilities', 'ability', 'ablaze', 'able',
       'abnormalities', 'aboard', 'abolish', 'abolished', 'abolition',
       'abolitionist'], dtype=object)

In [62]:
movie_index = 0

vector = tfidf_matrix[movie_index]

print("Movie:", df.iloc[movie_index]["Movie_Title"])
print("Vector shape:", vector.shape)
print("Non-zero values:", vector.nnz)

Movie: The Fall Guy
Vector shape: (1, 18700)
Non-zero values: 17


## 3. Storyline-Based Recommendation Function

The recommendation function accepts a user's storyline as input.

The input is transformed using the **same TF-IDF vectorizer fitted on the movie dataset**. The resulting query vector is then compared with all movie vectors using cosine similarity.

The movies are ranked by similarity score, and the top 5 results are returned.

In [63]:
from sklearn.metrics.pairwise import cosine_similarity

In [64]:
def recommend_movies(storyline, top_n=5):
    """
    Recommend movies based on storyline similarity.

    Parameters
    ----------
    storyline : str
        User-provided movie storyline or plot description.

    top_n : int, default=5
        Number of recommendations to return.

    Returns
    -------
    pandas.DataFrame
        Top recommended movies with similarity scores.
    """

    if not isinstance(storyline, str):
        raise TypeError("Storyline must be a string.")

    storyline = storyline.strip()

    if not storyline:
        raise ValueError("Storyline cannot be empty.")

    if top_n < 1:
        raise ValueError("top_n must be at least 1.")

    # Convert user input into the same TF-IDF feature space
    query_vector = tfidf.transform([storyline])

    # Compare query against all movie vectors
    similarity_scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    # Get indices of the highest-scoring movies
    top_indices = similarity_scores.argsort()[::-1][:top_n]

    recommendations = df.iloc[top_indices][
        [
            "Movie_Title",
            "Storyline_Raw",
            "Storyline_Processed"
        ]
    ].copy()

    recommendations["similarity_score"] = (
        similarity_scores[top_indices]
    )

    recommendations = recommendations.reset_index(drop=True)

    return recommendations

In [65]:
query = """
A young hero discovers a hidden magical world and must
fight a powerful enemy while protecting the people he loves.
"""

recommendations = recommend_movies(
    query,
    top_n=5
)

recommendations

,Movie_Title,Storyline_Raw,Storyline_Processed,similarity_score
0,The Real Full Monty,Amer discovers a magical lizard in the desert ...,amer discovers magical lizard desert brings lu...,0.238354
1,Footage,"Amidst World War I's chaos, a grieving father ...",amidst world war chaos grieving father turns h...,0.214780
2,Beyond the Gaze: Jule Campbell's Swimsuit Issue,"When a mysterious planet appears in the sky, a...",mysterious planet appears sky young father mus...,0.183743
3,The American Society of Magical Negroes,A young man is recruited into a secret society...,young man recruited secret society magical bla...,0.174934
4,Bagman,A college student's life changes when he meets...,college student life changes meets powerful wo...,0.171172


In [66]:
for i, row in recommendations.iterrows():
    print(f"{i + 1}. {row['Movie_Title']}")
    print(f"   Similarity: {row['similarity_score']:.4f}")
    print(f"   Storyline: {row['Storyline_Raw']}")
    print()

1. The Real Full Monty
   Similarity: 0.2384
   Storyline: Amer discovers a magical lizard in the desert that brings him luck, forcing him to choose between protecting it or selling out to those who want to exploit its powers.

2. Footage
   Similarity: 0.2148
   Storyline: Amidst World War I's chaos, a grieving father turns hero, leading villagers to safety while evading a relentless enemy driven by vengeance.

3. Beyond the Gaze: Jule Campbell's Swimsuit Issue
   Similarity: 0.1837
   Storyline: When a mysterious planet appears in the sky, a young father must choose between the life he loves and an ancient call to save the world.

4. The American Society of Magical Negroes
   Similarity: 0.1749
   Storyline: A young man is recruited into a secret society of magical black people who dedicate their lives to a cause of utmost importance: making white people's lives easier.

5. Bagman
   Similarity: 0.1712
   Storyline: A college student's life changes when he meets a powerful woman afte

In [67]:
action_query = """
An undercover agent investigates a dangerous criminal organization
and must survive a violent conspiracy.
"""

recommend_movies(action_query, top_n=5)

,Movie_Title,Storyline_Raw,Storyline_Processed,similarity_score
0,The Everything Pot,A drug enforcement agent teams up with her dau...,drug enforcement agent teams daughter save abd...,0.280271
1,Cholo Zombies,A wayward young man becomes involved in a crim...,wayward young man becomes involved criminal co...,0.274603
2,Beyond the Lake,The story focuses on an American agent sent to...,story focuses american agent sent dangerous un...,0.266854
3,Can't Let It Go,A secret agent realizes the organization he wo...,secret agent realizes organization works plans...,0.253896
4,Kill Em All 2,A former DEA agent and a former undercover ope...,former dea agent former undercover operative r...,0.251978


In [68]:
romance_query = """
Two people unexpectedly fall in love while dealing with family,
career, and difficult personal choices.
"""

recommend_movies(romance_query, top_n=5)

,Movie_Title,Storyline_Raw,Storyline_Processed,similarity_score
0,Electric Lady Studios: A Jimi Hendrix Vision,A husband's unwavering support for his wife's ...,husband unwavering support wife dreams leads p...,0.208035
1,LeGrand,Two women fall in love with the charms of a bl...,two women fall love charms black guy hurts lot...,0.183137
2,3 Obás de Xangô,An Internet-famous chef must rehabilitate her ...,internet famous chef must rehabilitate life ca...,0.175948
3,Youth (Homecoming),A documentary about dinosaurs and the people w...,documentary dinosaurs people love,0.171094
4,Ennio Doris - C'è anche domani,Long-feuding families unite through grandma's ...,long feuding families unite grandma wish arran...,0.163532


In [69]:
scifi_query = """
A group of astronauts face an unknown threat while trying to
survive aboard a damaged spacecraft far from Earth.
"""

recommend_movies(scifi_query, top_n=5)

,Movie_Title,Storyline_Raw,Storyline_Processed,similarity_score
0,Tenants,"In April 1970, NASA faced the greatest crisis ...",april nasa faced greatest crisis history three...,0.234452
1,The Jack in the Box Rises,A group of astronauts venture to a mysterious ...,group astronauts venture mysterious planet cla...,0.229194
2,Audrey,When meteor storms bring a new life form to ea...,meteor storms bring new life form earth global...,0.171613
3,A Kidnapping in Amish Country,Astronauts discover alarming evidence on a vac...,astronauts discover alarming evidence vacant p...,0.170749
4,Elevator,High school graduate Chloe babysits adopted da...,high school graduate chloe babysits adopted da...,0.159742


In [70]:
recommendations["similarity_score"].describe()

count    5.000000
mean     0.196597
std      0.028962
min      0.171172
25%      0.174934
50%      0.183743
75%      0.214780
max      0.238354
Name: similarity_score, dtype: float64

## Recommendation Logic

TF-IDF converts each movie storyline into a numerical vector representation.

For a user-provided storyline, the same fitted TF-IDF vectorizer converts the query into a vector in the same feature space.

Cosine Similarity then measures the similarity between the query vector and each movie vector.

The movies are ranked by similarity score, and the highest-scoring five movies are returned.

In [71]:
print(
    "Minimum score:",
    recommendations["similarity_score"].min()
)

print(
    "Maximum score:",
    recommendations["similarity_score"].max()
)

Minimum score: 0.17117179325054666
Maximum score: 0.2383542163329603


## Model-Data Alignment

The rows of the movie dataframe must remain in the same order as the rows of the TF-IDF matrix.

For example:

```text
DataFrame Row 0  ↔  TF-IDF Vector 0
DataFrame Row 1  ↔  TF-IDF Vector 1
DataFrame Row 2  ↔  TF-IDF Vector 2
...

## 4. Model Persistence

The fitted TF-IDF vectorizer and TF-IDF matrix are saved as model artifacts.

This allows the Streamlit application to load the existing recommendation representation directly instead of refitting the TF-IDF model every time the application starts.

In [72]:
import joblib

In [73]:
vectorizer_path = MODELS_DIR / "tfidf_vectorizer.pkl"

joblib.dump(
    tfidf,
    vectorizer_path
)

['G:\\Projects\\IMDb_Movie_Recommendation_System\\models\\tfidf_vectorizer.pkl']

In [74]:
matrix_path = MODELS_DIR / "tfidf_matrix.pkl"

joblib.dump(
    tfidf_matrix,
    matrix_path
)

['G:\\Projects\\IMDb_Movie_Recommendation_System\\models\\tfidf_matrix.pkl']

In [75]:
print("Vectorizer saved to:", vectorizer_path)
print("TF-IDF matrix saved to:", matrix_path)

Vectorizer saved to: G:\Projects\IMDb_Movie_Recommendation_System\models\tfidf_vectorizer.pkl
TF-IDF matrix saved to: G:\Projects\IMDb_Movie_Recommendation_System\models\tfidf_matrix.pkl


In [76]:
movie_data_path = MODELS_DIR / "movie_data.pkl"

df.to_pickle(movie_data_path)

print(
    "Movie data saved to:",
    movie_data_path
)

Movie data saved to: G:\Projects\IMDb_Movie_Recommendation_System\models\movie_data.pkl


In [77]:
print("Movie rows :", len(df))
print("TF-IDF rows:", tfidf_matrix.shape[0])

assert len(df) == tfidf_matrix.shape[0], (
    "Movie data and TF-IDF matrix are misaligned."
)

Movie rows : 6021
TF-IDF rows: 6021


## 5. Recommendation System Summary

The recommendation engine uses a content-based approach to compare a user-provided storyline with the storylines of movies in the dataset.

### Pipeline

```text
Processed Movie Storylines
          ↓
     TF-IDF Fitting
          ↓
    TF-IDF Matrix
          ↓
   User Storyline
          ↓
   TF-IDF Transform
          ↓
    Query Vector
          ↓
  Cosine Similarity
          ↓
   Similarity Ranking
          ↓
      Top 5 Movies

## Model Loader Testing:

In [78]:
#Testing src/preprocessing.py
from src.preprocessing import (
    clean_movie_title,
    normalize_text,
    tokenize_text,
    remove_stopwords,
    preprocess_storyline,
)

In [79]:
print(clean_movie_title("123. The Fall Guy"))

The Fall Guy


In [80]:
text = "A Young Hero discovers a magical world!"
print(normalize_text(text))

a young hero discovers a magical world


In [81]:
storyline = """
A young hero discovers a magical world and fights
against a powerful enemy!
"""

print(preprocess_storyline(storyline))

young hero discovers magical world fights powerful enemy


In [82]:
from src.model_loader import load_models
models = load_models()

In [83]:
print(models.keys())

dict_keys(['movie_data', 'tfidf_matrix', 'tfidf_vectorizer'])


In [84]:
movie_data = models["movie_data"]
tfidf_matrix = models["tfidf_matrix"]
tfidf_vectorizer = models["tfidf_vectorizer"]

In [85]:
print("Movie data type       :", type(movie_data).__name__)
print("TF-IDF matrix type    :", type(tfidf_matrix).__name__)
print("Vectorizer type       :", type(tfidf_vectorizer).__name__)

Movie data type       : DataFrame
TF-IDF matrix type    : csr_matrix
Vectorizer type       : TfidfVectorizer


In [86]:
print("Movie data shape:", movie_data.shape)
print("TF-IDF shape    :", tfidf_matrix.shape)

Movie data shape: (6021, 3)
TF-IDF shape    : (6021, 18700)


In [87]:
print(
    "Rows aligned:",
    len(movie_data) == tfidf_matrix.shape[0]
)

Rows aligned: True


In [88]:
movie_data.head()

,Movie_Title,Storyline_Raw,Storyline_Processed
0,The Fall Guy,"A stuntman, fresh off an almost career-ending ...",stuntman fresh almost career ending accident t...
1,The Substance,A fading celebrity takes a black-market drug: ...,fading celebrity takes black market drug cell ...
2,The Life of Chuck,"A life-affirming, genre-bending story about th...",life affirming genre bending story three chapt...
3,Abigail,After a group of criminals kidnap the ballerin...,group criminals kidnap ballerina daughter powe...
4,The Ministry of Ungentlemanly Warfare,The British military recruits a small group of...,british military recruits small group highly s...


In [89]:
print(movie_data.columns.tolist())

['Movie_Title', 'Storyline_Raw', 'Storyline_Processed']


In [90]:
print(movie_data["Storyline_Processed"].isna().sum())

0


In [91]:
test_query = "A young hero fights a powerful enemy to save his family."

query_vector = tfidf_vectorizer.transform([test_query])

print("Query vector shape:", query_vector.shape)
print("Non-zero features:", query_vector.nnz)

Query vector shape: (1, 18700)
Non-zero features: 7


In [92]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_scores = cosine_similarity(
    query_vector,
    tfidf_matrix
).flatten()

print("Number of similarity scores:", len(similarity_scores))
print("Highest score:", similarity_scores.max())

Number of similarity scores: 6021
Highest score: 0.25489006274040144


## Recommender Testing

In [93]:
from src.model_loader import load_models
from src.recommender import MovieRecommender

In [94]:
models = load_models()

movie_data = models["movie_data"]
tfidf_matrix = models["tfidf_matrix"]
tfidf_vectorizer = models["tfidf_vectorizer"]

In [95]:
recommender = MovieRecommender(
    movie_data=movie_data,
    tfidf_matrix=tfidf_matrix,
    tfidf_vectorizer=tfidf_vectorizer,
)

In [96]:
query = """
A group of astronauts faces an unknown creature
while trying to survive aboard a damaged spacecraft.
"""

recommendations = recommender.recommend(
    query,
    top_n=5,
)

recommendations

,Movie_Title,Storyline_Raw,Storyline_Processed,similarity_score
0,Tenants,"In April 1970, NASA faced the greatest crisis ...",april nasa faced greatest crisis history three...,0.250161
1,The Jack in the Box Rises,A group of astronauts venture to a mysterious ...,group astronauts venture mysterious planet cla...,0.244551
2,A Kidnapping in Amish Country,Astronauts discover alarming evidence on a vac...,astronauts discover alarming evidence vacant p...,0.212048
3,Monster Island,"Set in the Pacific, 1942. A Japanese soldier a...",set pacific japanese soldier british prisoner ...,0.183665
4,Bangsal Isolasi,A CIA special operations team out on demolitio...,cia special operations team demolition maneuve...,0.168975


In [97]:
raw_query = """
A Young Hero discovers a Magical World
and fights a powerful enemy!
"""

processed_query = preprocess_storyline(
    raw_query
)

print("Raw query:")
print(raw_query)

print("\nProcessed query:")
print(processed_query)

Raw query:

A Young Hero discovers a Magical World
and fights a powerful enemy!


Processed query:
young hero discovers magical world fights powerful enemy


## Error Testing

In [98]:
recommender.recommend("")

ValueError: Storyline cannot be empty.

In [99]:
recommender.recommend(query, top_n=0)

ValueError: top_n must be at least 1.

In [100]:
recommender.recommend(123)

TypeError: Storyline must be a string.

In [101]:
query = """
A Young Hero discovers a MAGICAL world!!!
He must fight a powerful enemy, protect his family,
and save everyone.
"""

print("Original:")
print(query)

print("\nRecommendations:")
print(
    recommender.recommend(
        query,
        top_n=5
    )[["Movie_Title", "similarity_score"]]
)

Original:

A Young Hero discovers a MAGICAL world!!!
He must fight a powerful enemy, protect his family,
and save everyone.


Recommendations:
                 Movie_Title  similarity_score
0                    Footage          0.242205
1               Space & Time          0.219712
2                Mother Mara          0.188810
3            A Strange House          0.173950
4  The Greatest Night in Pop          0.173260
